# Machine Learning Model Development

After completing Exploratory Data Analysis (EDA) and data preprocessing, the next step is to prepare the dataset for machine learning model development. The objective is to train and evaluate different machine learning algorithms for phishing URL detection using the processed PhiUSIIL dataset.

In [1]:
# Import Required Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

## Load the Preprocessed Dataset

The cleaned phishing URL dataset is loaded into the notebook. This dataset has already undergone data cleaning and preprocessing during the previous phase of the project.

In [2]:
df = pd.read_csv("cleaned_dataset.csv")

df.head()

,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1
3,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,...,1,1,1,3,27,15,22,1,31,1
4,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,...,1,0,1,244,15,34,72,1,85,1


## Feature Selection

The original dataset contains several textual attributes such as URL, Domain, TLD, and Title. Since tree-based machine learning algorithms require numerical input features, these textual columns are removed before model training.

The target variable is **label**, where:

- 0 → Legitimate Website
- 1 → Phishing Website

In [3]:
# Features
X = df.drop(columns=['URL', 'Domain', 'TLD', 'Title', 'label'])

# Target
y = df['label']

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (235795, 50)
Target Shape: (235795,)


## Train-Test Data Split

According to the reviewed research papers, an 80:20 train-test split provides a suitable balance between model learning and performance evaluation.

Stratified sampling is applied to preserve the original class distribution in both the training and testing datasets.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

Training Samples : (188636, 50)
Testing Samples  : (47159, 50)


# Random Forest Classification

Random Forest is an ensemble machine learning algorithm that constructs multiple decision trees and combines their predictions to improve classification performance and reduce overfitting.

It is widely used for phishing website detection because of its robustness, high accuracy, and ability to handle high-dimensional feature spaces.

In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Random Forest Prediction

In [6]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

## Random Forest Evaluation

The trained Random Forest model is evaluated using the following performance metrics:

- Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix
- ROC-AUC Score

In [7]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_rf))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Accuracy: 1.0

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159


Confusion Matrix

[[20189     0]
 [    0 26970]]

ROC-AUC: 1.0


## Cross Validation

To evaluate the robustness and generalization capability of the Random Forest model, 5-Fold Cross Validation is performed.

In [8]:
scores_rf = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print(scores_rf)
print("Mean Accuracy:", scores_rf.mean())

[1. 1. 1. 1. 1.]
Mean Accuracy: 1.0


## Feature Importance

Random Forest provides feature importance scores, allowing identification of the most influential features contributing to phishing website detection.

In [9]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

,Feature,Importance
3,URLSimilarityIndex,0.181075
49,NoOfExternalRef,0.169265
22,LineOfCode,0.146585
47,NoOfSelfRef,0.108536
44,NoOfImage,0.084350
46,NoOfJS,0.072904
45,NoOfCSS,0.031598
36,HasSocialNet,0.031526
43,HasCopyrightInfo,0.025522
21,IsHTTPS,0.022940


# XGBoost Classification

Extreme Gradient Boosting (XGBoost) is an advanced ensemble learning algorithm that improves prediction performance using gradient boosting techniques.

It is selected for comparison because of its high predictive accuracy and efficiency on structured datasets.

In [10]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

## XGBoost Prediction

In [11]:
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

## XGBoost Evaluation

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

Accuracy: 1.0

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159


Confusion Matrix

[[20189     0]
 [    0 26970]]

ROC-AUC: 1.0


# Logistic Regrassion

# Feature Scaling

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regrassion

In [17]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

# Prediction 

In [18]:
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

# Evaluation

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.9998727708390763
Precision: 0.9997775800711743
Recall   : 1.0
F1 Score : 0.9998887776665555
ROC AUC  : 0.9999999779613244

Confusion Matrix
[[20183     6]
 [    0 26970]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



# Decision Tree

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Train Decision Tree
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

# Prediction
y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)[:, 1]

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC AUC  : 1.0000

Confusion Matrix
[[20189     0]
 [    0 26970]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159



# All Algorithms Prediciton Scores 

In [22]:
# Logistic Regression
lr_pred = lr.predict(X_test_scaled)

# Decision Tree
dt_pred = dt.predict(X_test)

# Random Forest
rf_pred = rf.predict(X_test)

# XGBoost
xgb_pred = xgb.predict(X_test)

print("="*50)
print("Model Accuracy Comparison")
print("="*50)

print(f"Logistic Regression : {accuracy_score(y_test, lr_pred):.6f}")
print(f"Decision Tree       : {accuracy_score(y_test, dt_pred):.6f}")
print(f"Random Forest       : {accuracy_score(y_test, rf_pred):.6f}")
print(f"XGBoost             : {accuracy_score(y_test, xgb_pred):.6f}")

Model Accuracy Comparison
Logistic Regression : 0.999873
Decision Tree       : 1.000000
Random Forest       : 1.000000
XGBoost             : 1.000000


In [26]:
print(scores_xgb.mean())

0.9999957590279692


In [28]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ],
    "Cross Validation": [
        lr_scores.mean(),
        dt_scores.mean(),
        rf_scores.mean(),
        xgb_scores.mean()
    ]
})

comparison

,Model,Accuracy,Cross Validation
0,Logistic Regression,0.999873,0.999869
1,Decision Tree,1.000000,1.000000
2,Random Forest,1.000000,1.000000
3,XGBoost,1.000000,0.999996


## Conclusion

The PhiUSIIL Phishing URL Dataset was successfully cleaned and prepared
for machine learning model development.

During preprocessing, unnecessary textual attributes such as URL, Domain,
TLD, and Title were removed from the feature matrix, while the target
variable was separated as the label.

The dataset was transformed into a numerical feature matrix suitable for
machine learning algorithms. The input features were separated from the
target variable, and the processed dataset was saved as
`cleaned_dataset.csv`.

The resulting preprocessed dataset is now ready for the model training
phase, where different machine learning algorithms will be trained and
evaluated for phishing URL detection.